# Multivariate Gaussian Prior — toy example

This notebook demonstrates the `GaussianMultiVariatePrior`, which places
a correlated Gaussian constraint on a set of parameters.

A toy neutrino-flux model with two components is used:
- **astro** — astrophysical power-law flux parameterised by `astro_norm` and `astro_index`
- **atmo**  — atmospheric power-law flux parameterised by `atmo_norm`

`astro_norm` and `astro_index` are constrained by a bivariate Gaussian prior
with positive correlation ρ = 0.7, while `atmo_norm` has a flat (uniform) prior.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyForwardFolding as pyFF
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

## 1. Dataset creation

We simulate `n_events` neutrino-like events drawn from a flat distribution
in log-energy and uniform in cos(zenith). Reconstructed quantities are obtained
by smearing the true values with Gaussian resolution functions:
- Energy: σ = 0.2 in log10(E)
- Zenith:  σ = 0.05 in cos(θ)

Baseline weights convert event counts into a physical flux-weighted expectation.

In [ ]:
n_events = int(5E6)

event_energies = 10 ** np.random.uniform(2, 8, size=n_events)
event_zenith   = np.arccos(np.random.uniform(-1, 1, size=n_events))

reco_energy = 10 ** (np.log10(event_energies) + np.random.normal(0, 0.2, size=n_events))
reco_zenith = np.arccos(
    np.clip(np.cos(event_zenith) + np.random.normal(0, 0.05, size=n_events), -1, 1)
)

baseline_weights = event_energies * np.log(10) * 6 / n_events * 1e5

data = {"dataset": {
    "log10_reco_energy": jnp.log10(jnp.asarray(reco_energy)),
    "cos_reco_zenith":   jnp.cos(jnp.asarray(reco_zenith)),
    "baseline_weight":   jnp.asarray(baseline_weights),
    "true_energy":       jnp.asarray(event_energies),
    "true_zenith":       jnp.asarray(event_zenith),
}}

## 2. Load config and inspect the prior

In [ ]:
config = "./multivariate_prior_test.yaml"

ana            = pyFF.config.analysis_from_config(config)
params, priors = pyFF.config.params_from_config(config)

print("Priors:", priors)
print("Model parameters:", params)

In [ ]:
# The second prior is the GaussianMultiVariatePrior
mv_prior = priors[1]

print("param_names :", mv_prior.param_names)
print("mean        :", mv_prior.mean)
print("covariance  :\n", mv_prior.cov)

# Derive the correlation matrix from the covariance
stds = jnp.sqrt(jnp.diag(mv_prior.cov))
corr = mv_prior.cov / jnp.outer(stds, stds)
print("correlation :\n", corr)

## 3. Generate pseudo-data and plot component breakdown

In [ ]:
obs, _      = ana.evaluate(data, params)
cobs, _     = ana.evaluate_per_component(data, params)

fig, ax = plt.subplots()
ax.stairs(obs["det1"].sum(axis=1),               label="total")
ax.stairs(cobs["det1"]["astro"].sum(axis=1),     label="astro")
ax.stairs(cobs["det1"]["atmo"].sum(axis=1),      label="atmo")
ax.set_yscale("log")
ax.set_xlabel("log10(E) bin")
ax.set_ylabel("expected counts")
ax.legend()
plt.tight_layout()

## 4. Minimization

In [ ]:
llh  = pyFF.likelihood.PoissonLikelihood(ana, priors)
mini = pyFF.minimizer.ScipyMinimizer(llh)

result = mini.minimize(obs, data)
print(result)

## 5. 2D scan — visualising the correlated prior

We fix `atmo_norm` at its best-fit value and scan a grid of
`(astro_norm, astro_index)` values.

Two surfaces are shown side by side:
- **Left** — the prior log-pdf over the grid, directly showing the tilted ellipse
  from the ρ = 0.7 correlation between the two parameters.
- **Right** — the full Poisson log-likelihood including the prior, showing how the
  data and the prior combine.

> Note: a naive 'Poisson-only' surface cannot be obtained by subtraction — doing so
> would implicitly hold `atmo_norm` at its prior-informed best-fit, which is not the
> same optimum you would get from a prior-free fit.

In [ ]:
best_fit_pars = result[1]  # dict returned by ScipyMinimizer

norm_vals  = np.linspace(0.5, 2.5, 30)
index_vals = np.linspace(1.2, 2.8, 30)

llh_grid   = np.zeros((len(index_vals), len(norm_vals)))
prior_grid = np.zeros_like(llh_grid)

for i, idx in enumerate(index_vals):
    for j, nrm in enumerate(norm_vals):
        pv = dict(best_fit_pars)
        pv["astro_norm"]  = nrm
        pv["astro_index"] = idx
        llh_grid[i, j]   = float(llh.llh(obs, data, pv))
        prior_grid[i, j] = float(mv_prior.log_pdf(pv))

print("Scan complete")

In [ ]:
def delta_llh(grid):
    """2 * (max - grid) so the minimum sits at 0."""
    return 2 * (grid.max() - grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
levels = [1.0, 4.0, 9.0]  # ~1σ, 2σ, 3σ for 2 d.o.f.

for ax, grid, title in zip(
    axes,
    [delta_llh(prior_grid), delta_llh(llh_grid)],
    ["Prior only  (−2Δ log prior)", "Full LLH  (Poisson + prior)"],
):
    cf = ax.contourf(
        norm_vals, index_vals, grid,
        levels=np.linspace(0, 12, 40), cmap="viridis_r"
    )
    ax.contour(
        norm_vals, index_vals, grid,
        levels=levels, colors="white", linewidths=1.2
    )
    ax.set_xlabel("astro_norm")
    ax.set_ylabel("astro_index")
    ax.set_title(title)
    ax.axvline(mv_prior.mean[0], color="red", ls="--", lw=1, label="prior mean")
    ax.axhline(mv_prior.mean[1], color="red", ls="--", lw=1)
    ax.legend(fontsize=8)

fig.colorbar(cf, ax=axes, label="-2Δ log L")
plt.suptitle("Correlated prior (ρ = 0.7): prior ellipse vs full likelihood")
plt.tight_layout()